# Module 4 — Nettoyage de données : rendre un dataset exploitable

Dans un projet data réel, les données sont rarement propres dès le départ. Il peut y avoir des valeurs manquantes, des doublons, des textes mal écrits, des montants en format texte, des dates invalides ou des colonnes inutiles.

Nettoyer les données signifie corriger ces problèmes pour obtenir un tableau fiable. Sans nettoyage, une analyse peut produire de mauvais chiffres et conduire à de mauvaises décisions.

Exemple simple : si une même vente apparaît deux fois, le chiffre d’affaires sera trop élevé. Si un montant est écrit comme texte, le calcul peut échouer. Si une date est manquante, on peut mal analyser les tendances.

> **Objectif du module** : apprendre à repérer les problèmes courants, corriger les types, gérer les valeurs manquantes, supprimer les doublons et préparer un dataset propre.


## 1. Qu’est-ce qu’une donnée sale ?

Une donnée sale est une donnée qui peut empêcher l’analyse ou fausser le résultat.

Exemples fréquents :

| Problème | Exemple | Risque |
|---|---|---|
| Valeur manquante | client vide | impossible de relancer |
| Doublon | même commande deux fois | chiffre d’affaires gonflé |
| Type incorrect | `"29000"` au lieu de `29000` | calcul impossible ou faux |
| Texte incohérent | `whatsapp`, `WhatsApp`, ` whatsApp ` | regroupement incorrect |
| Date invalide | `32/13/2026` | analyse temporelle impossible |

> **À retenir** : nettoyer, ce n’est pas embellir les données. C’est les rendre fiables.


In [ ]:
donnees = [
    {"commande_id": "C001", "client": "Awa", "montant": "29000", "canal": " whatsapp ", "date": "2026-06-01"},
    {"commande_id": "C002", "client": "Moussa", "montant": "", "canal": "Facebook", "date": "2026-06-01"},
    {"commande_id": "C001", "client": "Awa", "montant": "29000", "canal": "WHATSAPP", "date": "2026-06-01"},
    {"commande_id": "C003", "client": "Sarah", "montant": "45000", "canal": "TikTok", "date": ""},
]

print("Nombre de lignes brutes :", len(donnees))
print(donnees[0])


Nombre de lignes brutes : 4
{'commande_id': 'C001', 'client': 'Awa', 'montant': '29000', 'canal': ' whatsapp ', 'date': '2026-06-01'}


## 2. Inspecter avant de corriger

Avant de nettoyer, il faut observer. Une erreur fréquente est de modifier les données sans comprendre le problème.

L’inspection consiste à regarder :

- le nombre de lignes ;
- les colonnes disponibles ;
- quelques exemples ;
- les valeurs manquantes ;
- les doublons possibles ;
- les formats de texte.

Cette étape évite de corriger au hasard.

> **Méthode simple** : regarder d’abord, décider ensuite, corriger enfin.


In [ ]:
colonnes = donnees[0].keys()
print("Colonnes :", list(colonnes))

for ligne in donnees:
    print(ligne)


## 3. Nettoyer les textes

Les textes peuvent contenir des espaces inutiles, des majuscules incohérentes ou des écritures différentes.

Exemple : `" whatsapp "`, `"WHATSAPP"` et `"WhatsApp"` désignent le même canal, mais Python peut les voir comme trois textes différents.

Pour corriger, on utilise souvent :

- `.strip()` pour enlever les espaces au début et à la fin ;
- `.lower()` pour mettre en minuscules ;
- parfois `.title()` pour mettre une forme lisible.

> **Pourquoi c’est important ?** Si les canaux sont mal écrits, les ventes WhatsApp seront divisées en plusieurs groupes.


In [ ]:
for ligne in donnees:
    ligne["canal"] = ligne["canal"].strip().lower()

for ligne in donnees:
    print(ligne["commande_id"], "->", ligne["canal"])


C001 -> whatsapp
C002 -> facebook
C001 -> whatsapp
C003 -> tiktok


## 4. Gérer les valeurs manquantes

Une valeur manquante est une information absente. Elle peut être vide (`""`), nulle (`None`) ou non renseignée.

Toutes les valeurs manquantes ne se corrigent pas de la même façon. Il faut choisir une règle claire.

Exemples de règles :

- si le client manque, on peut marquer `Client inconnu` ;
- si le montant manque, on ne peut pas calculer la vente ;
- si la date manque, on peut garder la ligne mais signaler le problème ;
- si une colonne est inutile, on peut l’ignorer.

> **Règle professionnelle** : ne jamais inventer une donnée importante sans le dire.


In [ ]:
lignes_valides = []
lignes_rejetees = []

for ligne in donnees:
    if ligne["montant"] == "":
        lignes_rejetees.append({"ligne": ligne, "raison": "montant manquant"})
    else:
        lignes_valides.append(ligne)

print("Lignes valides :", len(lignes_valides))
print("Lignes rejetées :", len(lignes_rejetees))
print(lignes_rejetees)


## 5. Convertir les types

Après avoir retiré les lignes qui n’ont pas de montant, on peut convertir le montant en nombre.

La conversion permet d’utiliser `sum`, de calculer une moyenne, de comparer des montants ou de faire un graphique.

Ici, `montant` passe de texte (`str`) à entier (`int`).

> **À retenir** : le nettoyage prépare les données pour l’analyse. Sans conversion correcte, l’analyse peut échouer.


In [ ]:
for ligne in lignes_valides:
    ligne["montant"] = int(ligne["montant"])

print(lignes_valides[0])
print("Type montant :", type(lignes_valides[0]["montant"]))


## 6. Supprimer les doublons

Un doublon est une ligne répétée. Dans notre exemple, la commande `C001` apparaît deux fois.

Si on garde les deux lignes, la vente d’Awa sera comptée deux fois. Le chiffre d’affaires sera faux.

Pour supprimer les doublons, il faut choisir une clé fiable. Ici, `commande_id` identifie une commande. Si le même `commande_id` revient, on garde la première ligne et on ignore les suivantes.

> **Attention** : supprimer les doublons doit toujours se faire avec une règle claire.


In [ ]:
commandes_vues = set()
donnees_uniques = []

for ligne in lignes_valides:
    identifiant = ligne["commande_id"]
    if identifiant not in commandes_vues:
        donnees_uniques.append(ligne)
        commandes_vues.add(identifiant)

print("Avant suppression doublons :", len(lignes_valides))
print("Après suppression doublons :", len(donnees_uniques))
print(donnees_uniques)


## 7. Documenter le nettoyage

Un nettoyage professionnel doit être traçable. Cela signifie qu’on doit pouvoir expliquer ce qui a été fait.

Exemple de résumé :

- espaces supprimés dans la colonne canal ;
- canaux mis en minuscules ;
- lignes sans montant retirées ;
- montants convertis en nombres ;
- doublons supprimés avec `commande_id`.

Cette documentation protège ton analyse. Si quelqu’un te demande pourquoi une ligne a disparu, tu peux répondre clairement.

> **À retenir** : une analyse sérieuse ne cache pas le nettoyage. Elle l’explique.


In [ ]:
resume_nettoyage = {
    "lignes_brutes": len(donnees),
    "lignes_rejetees": len(lignes_rejetees),
    "lignes_finales": len(donnees_uniques),
    "regles": [
        "canal nettoyé avec strip + lower",
        "lignes sans montant rejetées",
        "montant converti en int",
        "doublons supprimés avec commande_id",
    ]
}

print(resume_nettoyage)


## 8. Mini-projet guidé : dataset propre et chiffre fiable

Maintenant que les données sont nettoyées, on peut calculer un chiffre d’affaires plus fiable.

Le calcul doit se faire seulement après les corrections. Sinon, tu risques d’additionner des doublons ou des montants invalides.

Ce mini-projet montre une règle fondamentale : **la qualité du résultat dépend de la qualité des données**.


In [ ]:
chiffre_affaires = sum(ligne["montant"] for ligne in donnees_uniques)

print("Chiffre d'affaires nettoyé :", chiffre_affaires, "FCFA")
print("Nombre de commandes propres :", len(donnees_uniques))


## Erreurs fréquentes à éviter

### 1. Supprimer sans comprendre

Ne supprime pas une ligne juste parce qu’elle semble bizarre. Identifie d’abord le problème.

### 2. Remplacer un montant manquant par zéro sans réfléchir

Un montant manquant et un montant égal à zéro ne veulent pas dire la même chose.

### 3. Oublier les doublons

Les doublons peuvent gonfler les ventes et donner une fausse impression de performance.

### 4. Ne pas documenter les règles

Sans résumé du nettoyage, ton analyse devient difficile à défendre.

### 5. Corriger seulement ce qui se voit

Certaines erreurs sont invisibles au premier regard. Il faut contrôler les types, les formats et les valeurs.


## Checklist avant QCM

Avant de passer au QCM, vérifie que tu peux expliquer :

- ce qu’est une donnée sale ;
- pourquoi on inspecte avant de nettoyer ;
- comment nettoyer un texte avec `strip` et `lower` ;
- pourquoi un montant vide pose problème ;
- pourquoi convertir un montant en nombre ;
- comment détecter et supprimer un doublon ;
- pourquoi documenter les règles de nettoyage ;
- pourquoi un chiffre calculé avant nettoyage peut être faux.
